# Order Reviews Data Profiling


## Load the Data

In [1]:
import pandas as pd
from pathlib import Path

bronze_path = Path.cwd().parent / "data" / "bronze" / "parquet"
df_order_reviews = pd.read_parquet (bronze_path/"olist_order_reviews_dataset.parquet")

print("Dimensions :", df_order_reviews.shape)

# Check data types and non-null values
df_order_reviews.info()

df_order_reviews.isna().sum()

df_order_reviews.head(5)

Dimensions : (99224, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   review_id                99224 non-null  object
 1   order_id                 99224 non-null  object
 2   review_score             99224 non-null  int64 
 3   review_comment_title     11568 non-null  object
 4   review_comment_message   40977 non-null  object
 5   review_creation_date     99224 non-null  object
 6   review_answer_timestamp  99224 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,None,None,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,None,None,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,None,None,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,None,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,None,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


## Check Duplicates

In [5]:
# Check completely identical rows
print(
    "Lignes dupliquées :",
    df_order_reviews.duplicated().sum()
)

# Check repeated review IDs
print(
    "review_id dupliqués :",
    df_order_reviews["review_id"].duplicated().sum()
)

# Check repeated order IDs
print(
    "order_id dupliqués :",
    df_order_reviews["order_id"].duplicated().sum()
)

# Number of unique review IDs
print(
    "review_id uniques :",
    df_order_reviews["review_id"].nunique()
)

Lignes dupliquées : 0
review_id dupliqués : 814
order_id dupliqués : 551
review_id uniques : 98410


In [6]:
# Retrieve all rows with repeated review_id values
duplicate_reviews = df_order_reviews[
    df_order_reviews["review_id"].duplicated(keep=False)
]

# Sort by review_id to compare rows
duplicate_reviews = duplicate_reviews.sort_values("review_id")

# Display the first 20 rows
display(duplicate_reviews.head(20))

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,None,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,None,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,None,None,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,None,None,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,None,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,None,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,None,None,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,None,None,2018-03-02 00:00:00,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,None,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,None,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44


## Check Review ID and Order ID Pairs

In [8]:
# Check duplicates on the review_id + order_id combination
duplicate_pairs = df_order_reviews.duplicated(
    subset=["review_id", "order_id"]
).sum()

print(
    "Couples review_id + order_id dupliqués :",
    duplicate_pairs
)

# Display the affected rows, if any
display(
    df_order_reviews[
        df_order_reviews.duplicated(
            subset=["review_id", "order_id"],
            keep=False
        )
    ].head(10)
)

Couples review_id + order_id dupliqués : 0


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp


## Check Customer Ratings

In [9]:
# Display the rating distribution
print(df_order_reviews["review_score"].value_counts().sort_index())

# Check invalid ratings
invalid_ratings = ~df_order_reviews["review_score"].between(1, 5)

print("Notes invalides :", invalid_ratings.sum())

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64
Notes invalides : 0


## Check Missing Values

In [10]:
# Check missing values in each column
print(df_order_reviews.isna().sum())

# Check empty text or text containing only whitespace
for column in ["review_comment_title", "review_comment_message"]:
    empty_values = (
        df_order_reviews[column]
        .fillna("")
        .str.strip()
        .eq("")
    )

    print(f"{column} : {empty_values.sum()} valeurs vides ou manquantes")

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64
review_comment_title : 87658 valeurs vides ou manquantes
review_comment_message : 58274 valeurs vides ou manquantes


## Check Dates

In [11]:
# Convert both columns to datetime
df_order_reviews["review_creation_date"] = pd.to_datetime(
    df_order_reviews["review_creation_date"],
    errors="coerce"
)

df_order_reviews["review_answer_timestamp"] = pd.to_datetime(
    df_order_reviews["review_answer_timestamp"],
    errors="coerce"
)

# Check whether the response predates the review creation
date_anomalies = (
    df_order_reviews["review_answer_timestamp"]
    < df_order_reviews["review_creation_date"]
)

print("Nombre d'anomalies de dates :", date_anomalies.sum())

Nombre d'anomalies de dates : 0


In [12]:
# Check invalid dates after conversion
print(
    df_order_reviews[
        ["review_creation_date", "review_answer_timestamp"]
    ].isna().sum()
)

review_creation_date       0
review_answer_timestamp    0
dtype: int64
